In [4]:
import pandas as pd
import re
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from sklearn.model_selection import train_test_split

# Load Dataset
data = pd.read_csv('eng_-french.csv')

# Data Cleaning
def clean_sentence(sentence):
    if not isinstance(sentence, str):  # Ensure the input is a string
        sentence = str(sentence)
    sentence = re.sub(r'<.*?>', '', sentence)  # Remove HTML tags
    sentence = re.sub(r'[^a-zA-Z\s]', '', sentence)  # Remove special characters
    sentence = sentence.lower().strip()  # Convert to lowercase and strip whitespace
    return sentence

data['cleaned_english'] = data['English words/sentences'].apply(clean_sentence)
data['cleaned_french'] = data['French words/sentences'].apply(clean_sentence)

# Tokenization
english_tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")
french_tokenizer = Tokenizer(num_words=20000, oov_token="<OOV>")

english_tokenizer.fit_on_texts(data['cleaned_english'])
french_tokenizer.fit_on_texts(data['cleaned_french'])

english_sequences = english_tokenizer.texts_to_sequences(data['cleaned_english'])
french_sequences = french_tokenizer.texts_to_sequences(data['cleaned_french'])

# Padding
max_length_english = max(len(seq) for seq in english_sequences)
max_length_french = max(len(seq) for seq in french_sequences)

english_padded = pad_sequences(english_sequences, maxlen=max_length_english, padding='post')
french_padded = pad_sequences(french_sequences, maxlen=max_length_french, padding='post')

# Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(english_padded, french_padded, test_size=0.2, random_state=42)

# Model Architecture
embedding_dim = 256
units = 512

# Encoder
encoder_inputs = Input(shape=(max_length_english,))
encoder_embedding = Embedding(input_dim=len(english_tokenizer.word_index) + 1,
                               output_dim=embedding_dim)(encoder_inputs)
encoder_lstm = LSTM(units, return_state=True)
encoder_outputs, state_h, state_c = encoder_lstm(encoder_embedding)

# Decoder
decoder_inputs = Input(shape=(max_length_french,))
decoder_embedding = Embedding(input_dim=len(french_tokenizer.word_index) + 1,
                               output_dim=embedding_dim)(decoder_inputs)
decoder_lstm = LSTM(units, return_sequences=True, return_state=True)
decoder_outputs, _, _ = decoder_lstm(decoder_embedding, initial_state=[state_h, state_c])
decoder_dense = Dense(len(french_tokenizer.word_index) + 1, activation='softmax')
decoder_outputs = decoder_dense(decoder_outputs)

# Compile Model
model = Model([encoder_inputs, decoder_inputs], decoder_outputs)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

# Prepare Decoder Target Data
decoder_target_data = np.expand_dims(y_train, axis=-1)

# Train Model
model.fit([X_train, y_train], decoder_target_data, batch_size=128, epochs=5, validation_split=0.2)

# Save Model
model.save("english_to_french_translation.h5")

# Evaluate Model
def evaluate_model(model, X, y):
    y_target = np.expand_dims(y, axis=-1)
    loss, accuracy = model.evaluate([X, y], y_target)
    print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

# Evaluate on Test Data
evaluate_model(model, X_test, y_test)


Epoch 1/5
879/879 ━━━━━━━━━━━━━━━━━━━━ 442s 501ms/step - accuracy: 0.8902 - loss: 1.1620 - val_accuracy: 0.9579 - val_loss: 0.3266
Epoch 2/5
879/879 ━━━━━━━━━━━━━━━━━━━━ 445s 504ms/step - accuracy: 0.9661 - loss: 0.2657 - val_accuracy: 0.9823 - val_loss: 0.1506
Epoch 3/5
879/879 ━━━━━━━━━━━━━━━━━━━━ 443s 504ms/step - accuracy: 0.9851 - loss: 0.1226 - val_accuracy: 0.9901 - val_loss: 0.0854
Epoch 4/5
879/879 ━━━━━━━━━━━━━━━━━━━━ 501s 503ms/step - accuracy: 0.9919 - loss: 0.0654 - val_accuracy: 0.9938 - val_loss: 0.0543
Epoch 5/5
879/879 ━━━━━━━━━━━━━━━━━━━━ 502s 503ms/step - accuracy: 0.9953 - loss: 0.0363 - val_accuracy: 0.9957 - val_loss: 0.0378


1098/1098 ━━━━━━━━━━━━━━━━━━━━ 67s 61ms/step - accuracy: 0.9957 - loss: 0.0386
Test Loss: 0.038297876715660095, Test Accuracy: 0.9957370758056641
